# C1 — `kvsim calibrate` on a real GPU

**What this is.** The first run of `kvsim calibrate` against a live engine. It was built after H6
showed that a replica count measured on one GPU does not transfer to another, so users have to
measure their own — and until now it had only been tested against a mock engine. One vLLM
**0.30.0** server, **Qwen/Qwen3-4B on one Tesla T4**, with the engine's natural KV capacity.

The rules were committed before this notebook ran (`configs/c1_calibrate.yaml`, and a dated entry in
`docs/09-hypotheses.md`). **C1 holds only if both hold:**

| | Rule | Prediction |
|---|---|---|
| **C1a** | `kvsim calibrate` exits 0: its cache check is within 5% **and** all 48 requests' cached tokens equal the model's | exact: hit rate **0.7143** (9,840 of 13,776 tokens) |
| **C1b** | the KV capacity it reads equals the engine log's `GPU KV cache size` ÷ 16 | equal |

Reported, not tested: the fitted timings against Phase 3's, the capacity against 1,537, and the
wall-clock time.

The command comes from the package's wheel, **embedded in this notebook** (sha256 `ac5209ef2899…`,
built from commit `5a34c6d`), so this runs exactly the committed code.


### How to run (≈ 20 minutes)
1. Kaggle → *Accelerator: GPU T4 ×2* (one T4 is used) → *Internet: on*.
2. *Run All*. The first cells install vLLM and download ~8 GB of weights.
3. Download **`c1_results.zip`** from the output panel and bring it back to the repo.

If a cell stops with ⛔, the zip is still written by the verdict cell when it can be; bring it back
anyway — a stop is a result too.


In [1]:
import json

# ── Generated from configs/c1_calibrate.yaml by scripts/c1_prepare.py. Do not edit here.
MODEL = "Qwen/Qwen3-4B"
VLLM_PINNED = "0.30.0"
BLOCK_SIZE = 16
MAX_MODEL_LEN = 2048
MAX_NUM_SEQS = 256
MAX_NUM_BATCHED_TOKENS = 8192
GPU_MEM_UTIL = 0.85
SEED = 0
CHECK_SESSIONS = 16
CHECK_TURNS = 3
TOKEN_RANGE = [1000, 30000]
MAX_ERROR = 0.05
PRED = json.loads(r"""{"requests": 48, "queries": 13776, "hits": 9840, "hit_rate": 0.714286, "cached_tokens": [0, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272]}""")          # the registered prediction for the cache check
PHASE3 = json.loads(r"""{"name": "t4-qwen3-4b-vllm0.30.0", "prefill_base_s": 0.0, "prefill_per_token_s": 0.0007014400878738409, "decode_base_s": 0.05291123879934786, "decode_per_seq_s": 0.001232162611867379, "kv_cache_blocks": 1537, "block_size": 16, "max_num_seqs": 256, "gpu": "Tesla T4", "model": "Qwen/Qwen3-4B", "engine_version": "0.30.0", "calibrated": true, "calibration_error": 0.002901}""")      # Phase 3's fitted T4 profile, for the report

# ── Box settings. Safe to change: nothing predicted depends on them.
GPU_INDEX = 0
PORT = 8000
STARTUP_TIMEOUT_S = 1800      # the first start downloads ~8 GB of weights and compiles
OUT_DIR = "c1_results"


In [2]:
import json
import os
import pathlib
import platform
import re
import shutil
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

pathlib.Path(OUT_DIR).mkdir(exist_ok=True)


class Stop(Exception):
    """Halts Run-All with a reason. Download phase3_results.zip anyway."""


def sh(cmd, timeout=60):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True,
                              timeout=timeout, check=False).stdout.strip()
    except Exception as e:  # an unfamiliar box: any binary may be missing
        return f"<failed: {e}>"


ENV = {
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "cpu_count": os.cpu_count(),
    "gpus": sh("nvidia-smi --query-gpu=index,name,memory.total,compute_cap,driver_version "
               "--format=csv,noheader"),
    "kaggle": bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")),
}
for k, v in ENV.items():
    print(f"{k:10s} {v}")
if "T4" not in ENV["gpus"]:
    print("\n⚠️  Not a Tesla T4. Cache counts stay valid (they are hardware-independent), but "
          "every\n   timing from this run must be labelled with the GPU above, not as a T4 figure.")


date_utc   2026-09-28T09:05:53+00:00
platform   Linux-6.12.90+-x86_64-with-glibc2.35
python     3.12.13
cpu_count  4
gpus       0, Tesla T4, 15360 MiB, 7.5, 580.159.04
1, Tesla T4, 15360 MiB, 7.5, 580.159.04
kaggle     True


In [3]:
# PINNED: the predictions were made for this version, and Phase 0 v2 passed every check on it.
!pip install -q vllm==0.30.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 5.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 71.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 84.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 78.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 67.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 45.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 31.7 MB/s eta 0:00:00


In [4]:
# @@KVSIM-WHEEL@@ — kv-routing-sim, from its own wheel. Checked against the registered hash first.
import base64
import hashlib

_WHEEL = base64.b64decode("""...""")
if hashlib.sha256(_WHEEL).hexdigest() != "ac5209ef2899b89604d84e178119c9826cc44b0ea8e34df8aba7c934e81c877a":
    raise Stop("⛔ the embedded wheel does not match its registered hash")
_path = pathlib.Path(OUT_DIR) / "kv_routing_sim-0.1.0-py3-none-any.whl"
_path.write_bytes(_WHEEL)
print(sh(f"{sys.executable} -m pip install -q {_path}", timeout=600) or "installed")
print(sh(f"{sys.executable} -m kv_routing_sim.cli calibrate --help | head -1"))


installed
usage: kvsim calibrate [-h] [--engine ENGINE] --out OUT [--model MODEL]


In [5]:
import importlib.metadata

import vllm

VLLM_VERSION = vllm.__version__
if VLLM_VERSION != VLLM_PINNED:
    raise Stop(f"⛔ vllm {VLLM_VERSION} is imported, but the predictions were made for "
               f"{VLLM_PINNED}. Restart the kernel and run from the top.")


def _version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None


ENV["packages"] = {p: _version(p) for p in ("vllm", "torch", "transformers", "triton")}
print("packages:", ENV["packages"])

ROOT = pathlib.Path(vllm.__file__).parent
SOURCES = {}
for p in ROOT.rglob("*.py"):
    try:
        SOURCES[str(p.relative_to(ROOT))] = p.read_text(errors="replace")
    except OSError:
        pass


def found_in(needle):
    return next((f for f, text in SOURCES.items() if needle in text), None)


# (what the simulator relies on, the exact source text in vLLM that implements it)
CITED = [
    ("null block reserved at start-up", "self.null_block = self.free_block_queue.popleft()"),
    ("uncached blocks reused first (prepend)", "prepend_n"),
    ("finished requests freed tail-first", "Free blocks in reverse order"),
    ("lookup capped at prompt_length - 1", "max_cache_hit_length = request.num_tokens - 1"),
    ("queries counted at admission only",
     "Record at admission so unscheduled lookups are not counted"),
    ("eviction event fired before hash check", "on_block_evicted(block)"),
    ("block access history bounded to 4", "deque(maxlen=4)"),
    ("reset also clears block metrics", "self.metrics_collector.reset()"),
    ("async scheduler caches on each output",
     "request, request.num_computed_tokens - request.num_output_placeholders"),
    ("full-prompt admission check, default on", "scheduler_reserve_full_isl: bool = True"),
    ("admission watermark, default 0", "watermark: float = Field(default=0.0"),
]
# (what this notebook sends or reads, the source text that serves it)
USED = [
    ("metric vllm:prefix_cache_queries", "vllm:prefix_cache_queries"),
    ("metric vllm:prefix_cache_hits", "vllm:prefix_cache_hits"),
    ("metric vllm:prompt_tokens_by_source", "vllm:prompt_tokens_by_source"),
    ("metric vllm:num_preemptions", "vllm:num_preemptions"),
    ("metric vllm:kv_block_lifetime_seconds", "vllm:kv_block_lifetime_seconds"),
    ("metric vllm:kv_block_idle_before_evict_seconds", "vllm:kv_block_idle_before_evict_seconds"),
    ("metric vllm:kv_block_reuse_gap_seconds", "vllm:kv_block_reuse_gap_seconds"),
    ("metric vllm:num_requests_running", "vllm:num_requests_running"),
    ("flag --num-gpu-blocks-override", "num_gpu_blocks_override"),
    ("flag --enable-prompt-tokens-details", "enable_prompt_tokens_details"),
    ("request field ignore_eos", "ignore_eos"),
    ("POST /reset_prefix_cache", '"/reset_prefix_cache"'),
    ("GET /server_info", '"/server_info"'),
]
AUDIT = {}
print(f"\nvllm {VLLM_VERSION}: {len(SOURCES):,} source files read")
for group, rows in (("the simulator cites", CITED), ("this notebook uses", USED)):
    print(f"── {group}")
    for what, needle in rows:
        AUDIT[what] = found_in(needle)
        print(f"  {'OK' if AUDIT[what] else '--'}  {what:44s} {AUDIT[what] or 'NOT FOUND'}")
AUDIT["vllm:prefix_cache_hit_rate (must NOT exist)"] = found_in('"vllm:prefix_cache_hit_rate"')
print(f"  {'OK' if not AUDIT['vllm:prefix_cache_hit_rate (must NOT exist)'] else '!!'}  "
      "vllm:prefix_cache_hit_rate does not exist (ADR-008)")

missing_used = [w for w, _ in USED if AUDIT[w] is None]
missing_cited = [w for w, _ in CITED if AUDIT[w] is None]
if missing_used:
    raise Stop(f"⛔ this vLLM lacks what the notebook needs: {missing_used}")
if missing_cited:
    print(f"\n⚠️  not found verbatim: {missing_cited}. Run anyway — the counts will show whether "
          "the behaviour itself changed.")


packages: {'vllm': '0.30.0', 'torch': '2.13.0', 'transformers': '5.17.0', 'triton': '3.7.1'}

vllm 0.30.0: 2,681 source files read
── the simulator cites
  OK  null block reserved at start-up              v1/core/block_pool.py
  OK  uncached blocks reused first (prepend)       v1/core/block_pool.py
  OK  finished requests freed tail-first           v1/core/single_type_kv_cache_manager.py
  OK  lookup capped at prompt_length - 1           v1/core/kv_cache_manager.py
  OK  queries counted at admission only            v1/core/sched/scheduler.py
  OK  eviction event fired before hash check       v1/core/block_pool.py
  OK  block access history bounded to 4            v1/core/kv_cache_metrics.py
  OK  reset also clears block metrics              v1/core/block_pool.py
  OK  async scheduler caches on each output        v1/core/sched/async_scheduler.py
  OK  full-prompt admission check, default on      config/scheduler.py
  OK  admission watermark, default 0               config/scheduler.py
─

In [6]:
BASE = f"http://127.0.0.1:{PORT}"
SERVER = {}
RESET_MODE = "endpoint"  # falls back to "restart" if the dev endpoint is not served
ENGINE_ARGV = [
    "vllm", "serve", MODEL, "--port", str(PORT),
    "--dtype", "half",  # T4 has no bf16
    "--block-size", str(BLOCK_SIZE),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--enable-prefix-caching",
    "--kv-cache-metrics", "--kv-cache-metrics-sample", "1.0",
    "--enable-prompt-tokens-details",
    "--seed", "0",
]


def port_in_use():
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", PORT)) == 0


def start_server(tag):
    # ⚠️ Refuse to launch while ANYTHING listens on the port. Otherwise a stale server (a cell
    # re-run without the clean-up) answers /health before the new process has had time to die
    # of "address already in use", and the run measures the wrong engine. Found by the dry-run:
    # checking our process first narrows that race but does not close it. A few seconds' grace
    # lets our own just-stopped server release the port.
    for _ in range(30):
        if not port_in_use():
            break
        time.sleep(1)
    else:
        raise Stop(f"⛔ something is already serving on port {PORT} — a server from an earlier "
                   "run? Run the clean-up cell (or !pkill -f 'vllm serve') and start again.")
    # VLLM_SERVER_DEV_MODE serves /reset_prefix_cache and /server_info.
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(GPU_INDEX), VLLM_SERVER_DEV_MODE="1")
    log_path = pathlib.Path(OUT_DIR) / f"server_{tag}.log"
    logf = open(log_path, "w")  # noqa: SIM115 - must outlive this cell
    proc = subprocess.Popen(ENGINE_ARGV, stdout=logf, stderr=subprocess.STDOUT, env=env)
    SERVER.clear()
    SERVER.update(proc=proc, logf=logf, log_path=str(log_path))
    print("launched:", " ".join(ENGINE_ARGV), f"  (GPU {GPU_INDEX})")


def log_text():
    return pathlib.Path(SERVER["log_path"]).read_text(errors="replace") if SERVER else ""


def http(method, path, payload=None, timeout=60):
    data = None if payload is None else json.dumps(payload).encode()
    req = urllib.request.Request(BASE + path, data=data, method=method,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, r.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8", errors="replace")
    except Exception as e:
        return 0, repr(e)


def wait_ready(timeout=STARTUP_TIMEOUT_S, every=30):
    """Ready = OUR process is alive AND the port answers, checked in that order: a stale server
    left on the port must never pass for ours (found by the Phase-0 dry-run)."""
    t0, last = time.time(), 0.0
    while time.time() - t0 < timeout:
        if SERVER["proc"].poll() is not None:
            print(f"  !! the server EXITED (returncode {SERVER['proc'].returncode}). Log tail:")
            print(log_text()[-3000:])
            if "address already in use" in log_text().lower():
                print("  -> another server owns the port: run the clean-up cell, or "
                      "!pkill -f 'vllm serve'")
            return False
        if http("GET", "/health", timeout=5)[0] == 200:
            print(f"  ready in {time.time() - t0:.0f}s")
            return True
        if time.time() - t0 - last >= every:
            last = time.time() - t0
            tail = log_text().strip().splitlines()
            print(f"    ...{last:.0f}s  {tail[-1][:110] if tail else '(log empty)'}")
        time.sleep(5)
    print(f"  !! not ready within {timeout}s")
    return False


def stop_server():
    if not SERVER:
        return
    SERVER["proc"].terminate()
    try:
        SERVER["proc"].wait(timeout=90)
    except subprocess.TimeoutExpired:
        SERVER["proc"].kill()
    SERVER["logf"].close()
    SERVER.clear()


def scrape():
    return http("GET", "/metrics", timeout=30)[1]


def metric(text, family, **labels):
    """Sum a family over matching labels. None = ABSENT, which is not the same as 0.
    prometheus_client appends _total to counters, so both spellings match."""
    lab = "".join(f'(?=[^}}]*{k}="{v}")' for k, v in labels.items())
    pat = re.compile(r"^" + re.escape(family) + r"(?:_total)?\{" + lab + r"[^}]*\}\s+(\S+)",
                     re.MULTILINE)
    vals = [float(x) for x in pat.findall(text)]
    return sum(vals) if vals else None


COUNTERS = {  # our name -> (metric family, labels)
    "prefix_cache_queries": ("vllm:prefix_cache_queries", {}),
    "prefix_cache_hits": ("vllm:prefix_cache_hits", {}),
    "local_compute": ("vllm:prompt_tokens_by_source", {"source": "local_compute"}),
    "local_cache_hit": ("vllm:prompt_tokens_by_source", {"source": "local_cache_hit"}),
    "preemptions": ("vllm:num_preemptions", {}),
    "hist_lifetime_count": ("vllm:kv_block_lifetime_seconds_count", {}),
    "hist_idle_count": ("vllm:kv_block_idle_before_evict_seconds_count", {}),
    "hist_reuse_gap_count": ("vllm:kv_block_reuse_gap_seconds_count", {}),
    "prompt_tokens": ("vllm:prompt_tokens", {}),
    "generation_tokens": ("vllm:generation_tokens", {}),
}


def read_counters(text):
    return {k: metric(text, fam, **lab) for k, (fam, lab) in COUNTERS.items()}


def settled_scrape(max_wait=180):
    """Scrape once nothing is running or waiting AND the counters have stopped moving. Stats
    reach /metrics a moment after the tokens they describe; an early read under-counts."""
    t0, prev = time.time(), None
    while True:
        text = scrape()
        busy = ((metric(text, "vllm:num_requests_running") or 0)
                + (metric(text, "vllm:num_requests_waiting") or 0))
        now = read_counters(text)
        if busy == 0 and now == prev:
            return text
        if time.time() - t0 > max_wait:
            raise Stop(f"⛔ the engine never settled within {max_wait}s (busy={busy})")
        prev = now
        time.sleep(1.0)


def deltas(before, after):
    b, a = read_counters(before), read_counters(after)
    return {k: None if a[k] is None else round(a[k] - (b[k] or 0.0)) for k in COUNTERS}


def fresh_cache(tag):
    """A cold cache before every run. The reset clears the hashes AND the block-metrics collector
    (block_pool.py reset_prefix_cache -> metrics_collector.reset()), so for every count predicted
    here a reset engine equals a freshly started one — without a 3-5 minute restart."""
    global RESET_MODE
    if RESET_MODE == "endpoint":
        for _ in range(15):
            code, body = http("POST", "/reset_prefix_cache")
            try:
                ok = code == 200 and json.loads(body).get("success") is True
            except ValueError:
                ok = False
            if ok:
                return
            if code in (404, 405):
                print("  /reset_prefix_cache is not served here: restarting the engine per run")
                RESET_MODE = "restart"
                break
            time.sleep(2)  # success=false: a finishing request still holds blocks
        else:
            raise Stop("⛔ the engine refused to reset its prefix cache 15 times")
    stop_server()
    start_server(tag)
    if not wait_ready():
        raise Stop("⛔ the restart failed — the log tail is above")


print("server helpers ready")


server helpers ready


In [7]:
start_server("c1")
if not wait_ready():
    raise Stop("⛔ the engine did not start — the log tail is above")
_m = re.search(r"GPU KV cache size: ([\d,]+) tokens", log_text())
LOG_BLOCKS = int(_m.group(1).replace(",", "")) // BLOCK_SIZE if _m else None
print(f"engine log: {LOG_BLOCKS:,} blocks of {BLOCK_SIZE} tokens" if LOG_BLOCKS
      else "⚠️  no 'GPU KV cache size' line in the engine log: C1b cannot be decided")


launched: vllm serve Qwen/Qwen3-4B --port 8000 --dtype half --block-size 16 --max-model-len 2048 --max-num-seqs 256 --max-num-batched-tokens 8192 --gpu-memory-utilization 0.85 --enable-prefix-caching --kv-cache-metrics --kv-cache-metrics-sample 1.0 --enable-prompt-tokens-details --seed 0   (GPU 0)
    ...30s  (log empty)
    ...60s  (APIServer pid=217) INFO 09-28 09:11:56 [kernel.py:416] Final IR op priority after setting platform defaults: 
    ...90s  (EngineCore pid=292) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to
    ...120s  (EngineCore pid=292) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to
    ...150s  (EngineCore pid=292) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to
    ...180s  (EngineCore pid=292) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to
    ...210s  (EngineCore pid=292) Warning: You are sending 

In [8]:
PROFILE_PATH = pathlib.Path(OUT_DIR) / "calibrated_profile.json"
CMD = [sys.executable, "-m", "kv_routing_sim.cli", "calibrate", "--engine", BASE,
       "--model", MODEL, "--out", str(PROFILE_PATH), "--seed", str(SEED),
       "--check-sessions", str(CHECK_SESSIONS), "--check-turns", str(CHECK_TURNS),
       "--max-error", str(MAX_ERROR), "--token-range", f"{TOKEN_RANGE[0]},{TOKEN_RANGE[1]}"]
print("$", " ".join(CMD))
_t0 = time.time()
_proc = subprocess.run(CMD, capture_output=True, text=True, timeout=3600, check=False)
WALL_S = round(time.time() - _t0, 1)
RC = _proc.returncode
pathlib.Path(OUT_DIR, "calibrate_stdout.txt").write_text(_proc.stdout)
pathlib.Path(OUT_DIR, "calibrate_stderr.txt").write_text(_proc.stderr)
print(_proc.stdout)
print(_proc.stderr)
print(f"exit code {RC} · {WALL_S} s")


$ /usr/bin/python3 -m kv_routing_sim.cli calibrate --engine http://127.0.0.1:8000 --model Qwen/Qwen3-4B --out c1_results/calibrated_profile.json --seed 0 --check-sessions 16 --check-turns 3 --max-error 0.05 --token-range 1000,30000
engine   http://127.0.0.1:8000 · vLLM 0.30.0 · Qwen/Qwen3-4B · GPU Tesla T4
         KV capacity: 1,537 blocks, from vllm:cache_config_info in /metrics
prefill  1.096 ms per uncached token + 0.0 ms  (R² 0.96, 24 requests)
decode   41.4 ms + 0.18 ms × batch  (R² 0.06, batches 1 2 4 8 16)
         fitted prefill base was -211.71 ms (< 0, not physical); clamped to 0 — the decode fit holds the whole fixed cost, as on the T4
check    hit rate: engine 0.7143, model 0.7143 → error 0.00% over 48 requests
         per-request cached tokens: 48 of 48 equal
wrote c1_results/calibrated_profile.json — calibrated. Use it: kvsim replay <trace> --profile c1_results/calibrated_profile.json


exit code 0 · 47.5 s


In [9]:
import zipfile

PROFILE = json.loads(PROFILE_PATH.read_text()) if PROFILE_PATH.exists() else None
_chk = (PROFILE or {}).get("_source", {}).get("check", {})
_eq, _cmp, _n = (_chk.get("cached_tokens_equal"), _chk.get("cached_tokens_compared"),
                 _chk.get("requests"))
C1A = bool(RC == 0 and PROFILE and PROFILE["calibrated"] and _eq == _cmp == _n)
C1B = bool(PROFILE and LOG_BLOCKS and PROFILE["kv_cache_blocks"] == LOG_BLOCKS)
_pred = _chk.get("predicted", {})
C1 = {
    "holds": C1A and C1B,
    "c1a": C1A,
    "c1b": C1B,
    # The embedded code must predict exactly the registered numbers; if not, the wheel is not the
    # code that made the prediction, and nothing here can be compared.
    "prediction_matches_registration": (_pred.get("hits"), _pred.get("queries"))
    == (PRED["hits"], PRED["queries"]),
    "returncode": RC,
    "measured": _chk.get("measured"),
    "predicted": _pred,
    "registered_prediction": PRED,
    "cached_tokens_equal": _eq,
    "cached_tokens_compared": _cmp,
    "log_blocks": LOG_BLOCKS,
    "read_blocks": PROFILE and PROFILE["kv_cache_blocks"],
    "reported": {
        "fitted": {k: PROFILE and PROFILE[k] for k in
                   ("prefill_base_s", "prefill_per_token_s", "decode_base_s", "decode_per_seq_s")},
        "phase3": {k: PHASE3[k] for k in
                   ("prefill_base_s", "prefill_per_token_s", "decode_base_s", "decode_per_seq_s",
                    "kv_cache_blocks")},
        "wall_clock_s": WALL_S,
    },
    "env": ENV,
    "audit": AUDIT,
}
pathlib.Path(OUT_DIR, "c1.json").write_text(json.dumps(C1, indent=1, default=str))
with zipfile.ZipFile("c1_results.zip", "w", zipfile.ZIP_DEFLATED) as _z:
    for _f in sorted(pathlib.Path(OUT_DIR).glob("*")):
        if not _f.name.endswith(".whl"):
            _z.write(_f)
print(f"C1a  calibrated, every request equal ... {'✅' if C1A else '⛔'}  "
      f"({_eq}/{_cmp} equal, rc {RC})")
print(f"C1b  capacity read = engine log ........ {'✅' if C1B else '⛔'}  "
      f"({C1['read_blocks']} read, {LOG_BLOCKS} in the log)")
print(f"prediction matches the registration ... {C1['prediction_matches_registration']}")
print(f"\n==> C1 {'HOLDS' if C1['holds'] else 'DOES NOT HOLD'} — bring back c1_results.zip")


C1a  calibrated, every request equal ... ✅  (48/48 equal, rc 0)
C1b  capacity read = engine log ........ ✅  (1537 read, 1537 in the log)
prediction matches the registration ... True

==> C1 HOLDS — bring back c1_results.zip


In [11]:
stop_server()
print("server stopped")


server stopped
